# Run Analysis: `20260122_210006`

This notebook loads artifacts under `runs/20260122_210006/` and summarizes Stage2→Stage4 outputs.

- Run dir: `workspace/kms/01_15_new_qlib/runs/20260122_210006`
- Main specs: `specs/*.json`
- Main data: `data/*.parquet`


In [14]:
from pathlib import Path
import sys

# If your notebook CWD is `.../01_15_new_qlib/analysis`, project root is the parent.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "analysis" else cwd
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.run_analyzer import (
    load_run,
    parse_run_log_summary,
    stage2_results_df,
    refinement_history_df,
    stage3_passed_combinations_df,
    stage3_combo_summary_df,
    stage4_combinations_df,
    read_parquet_any,
)

RUN_DIR = PROJECT_ROOT / "runs" / "20260122_210006"
assert RUN_DIR.exists(), f"Run dir not found: {RUN_DIR}"
RUN_DIR

PosixPath('/home/user/fin/FinAgent_4090/workspace/kms/01_15_new_qlib/runs/20260122_210006')

In [15]:
run = load_run(RUN_DIR)
rp = run["paths"]

log_summary = parse_run_log_summary(run.get("log_text", ""))
log_summary

{'n_stage2_iterations': 0,
 'stage2_iterations': [],
 'stage3_verdict': None,
 'stage4_started': True}

In [16]:
# High-level run metadata
hyp = run.get("hypothesis") or {}
split = run.get("data_split") or {}

print("hypothesis_id:", hyp.get("hypothesis_id"))
print("horizon_days:", hyp.get("horizon_days"))
print("IS:", split.get("in_sample_start"), "~", split.get("in_sample_end"))
print("OOS:", split.get("out_sample_start"), "~", split.get("out_sample_end"))


hypothesis_id: None
horizon_days: None
IS: 2023-01-01 ~ 2024-12-31
OOS: 2025-01-01 ~ 2025-12-31


## Stage2

- `stage2_summary.json` contains per-formula distribution evidence and PASS/FAIL verdict.


In [17]:
s2 = run.get("stage2_summary")
s2_df = stage2_results_df(s2)
print("stage2 overall:", s2.get("overall_verdict"), "pass_rate=", s2.get("pass_rate"))
s2_df.loc[:, ["formula_id", "observation_id", "verdict", "reasoning"]].head(10)


stage2 overall: FAIL pass_rate= 0.6


,formula_id,observation_id,verdict,reasoning
0,panic_selling_pressure_1,obs_panic_selling_pressure,PASS,DIR mean decreases from 0.056606 (Q1) to -0.04...
1,panic_selling_pressure_2,obs_panic_selling_pressure,PASS,DIR mean decreases from 0.010699 (Q1) to -0.01...
2,emotional_response_retail_1,obs_emotional_response_retail,PASS,VOL mean increases from 1072170.625 (Q1) to 18...
3,emotional_response_retail_2,obs_emotional_response_retail,PASS,MAG mean increases from 0.023745 (Q1) to 0.477...
4,leveraged_margin_calls_1,obs_leveraged_margin_calls,FAIL,The observation suggests exacerbating downward...
5,leveraged_margin_calls_2,obs_leveraged_margin_calls,FAIL,The observation suggests exacerbating downward...
6,institutional_accumulation_1,obs_institutional_accumulation,PASS,VOL mean increases from 1019373.4375 (Q1) to 1...
7,institutional_accumulation_2,obs_institutional_accumulation,PASS,DIR mean increases from -0.048215 (Q1) to 0.05...
8,price_displacement_1,obs_price_displacement,FAIL,The MAG mean shows an increasing pattern from ...
9,price_displacement_2,obs_price_displacement,FAIL,The observation suggests significant price dis...


## Refinement loop

- `refinement_history.json` shows which formulas failed and what feedback got injected.


In [18]:
rh = run.get("refinement_history")
rh_df = refinement_history_df(rh)
rh_df.head(20)


,iteration,n_total,n_passed,n_failed,pass_rate,failed_formulas
0,1,15,9,6,0.60,"[high_volume_1, high_volume_2, high_volume_3, ..."
1,2,12,9,3,0.75,"[panic_selling_pressure_3, price_below_intrins..."
2,3,10,6,4,0.60,"[leveraged_margin_calls_1, leveraged_margin_ca..."


## Stage3

- `stage3_result.json`: final verdict + the 5 qualified combinations passed to Stage4
- `stage3_ticker_details.json`: per-ticker evaluation details (used to recompute pass_rate / S2-ratio improvement)


In [19]:
s3 = run.get("stage3_result") or {}
print("stage3 verdict:", s3.get("overall_verdict"), "pass_rate=", s3.get("pass_rate"))

s3_passed = stage3_passed_combinations_df(s3)
s3_passed


stage3 verdict: PASS pass_rate= 0.6782608695652174


,instance_id,formula_names,combo_key
0,BH_MR_Panic_Reversion_3D_v1_instance_001,"[emotional_response_retail_1, institutional_ac...",emotional_response_retail_1|institutional_accu...
1,BH_MR_Panic_Reversion_3D_v1_instance_002,"[emotional_response_retail_1, institutional_ac...",emotional_response_retail_1|institutional_accu...
2,BH_MR_Panic_Reversion_3D_v1_instance_003,"[emotional_response_retail_2, institutional_ac...",emotional_response_retail_2|institutional_accu...
3,BH_MR_Panic_Reversion_3D_v1_instance_004,"[emotional_response_retail_2, institutional_ac...",emotional_response_retail_2|institutional_accu...
4,BH_MR_Panic_Reversion_3D_v1_instance_005,"[emotional_response_retail_2, institutional_ac...",emotional_response_retail_2|institutional_accu...


In [20]:
s3_details = run.get("stage3_ticker_details")
s3_combo_summary = stage3_combo_summary_df(s3_details)
s3_combo_summary.head(20)


,combo_key,n_eval,n_pass,pass_rate,avg_confidence,s2_ratio_first,s2_ratio_last,s2_ratio_improvement
7,emotional_response_retail_2|institutional_accu...,690,187,0.271014,0.216063,0.551399,0.433333,0.118065
1,emotional_response_retail_1|institutional_accu...,690,165,0.239130,0.189493,0.553900,0.466667,0.087233
0,emotional_response_retail_1|institutional_accu...,690,152,0.220290,0.174728,0.557603,0.574371,-0.016767
3,emotional_response_retail_1|institutional_accu...,690,148,0.214493,0.165278,0.552047,0.250000,0.302047
5,emotional_response_retail_2|institutional_accu...,690,121,0.175362,0.137530,0.553649,0.500000,0.053649
4,emotional_response_retail_2|institutional_accu...,690,90,0.130435,0.104076,0.556929,0.680000,-0.123071
2,emotional_response_retail_1|institutional_accu...,690,80,0.115942,0.087953,0.555654,0.750000,-0.194346
6,emotional_response_retail_2|institutional_accu...,690,68,0.098551,0.075362,0.554548,0.333333,0.221215


## Stage4

- `stage4_summary.json` has per-combination IS/OOS metrics (Sharpe, returns, drawdown, trades).
- `data/stage4_trades.parquet` has per-trade records.
- `data/stage4_positions.parquet` has daily per-ticker positions (if native Qlib backtest ran).


In [21]:
s4 = run.get("stage4_summary")
s4_df = stage4_combinations_df(s4)
s4_df.loc[:, [
    "combo_idx", "formula_names", "is_sharpe", "oos_sharpe",
    "is_net_return", "oos_net_return", "n_trades", "win_rate", "profit_factor",
]].sort_values("oos_sharpe", ascending=False)


,combo_idx,formula_names,is_sharpe,oos_sharpe,is_net_return,oos_net_return,n_trades,win_rate,profit_factor
4,5,"[emotional_response_retail_2, institutional_ac...",0.346567,1.464171,0.106758,0.267963,7908,0.439176,1.344769
3,4,"[emotional_response_retail_2, institutional_ac...",0.803786,1.206619,0.367365,0.222386,5569,0.418028,1.250521
0,1,"[emotional_response_retail_1, institutional_ac...",0.783761,0.346193,0.349875,0.044970,4052,0.443978,1.412755
2,3,"[emotional_response_retail_2, institutional_ac...",0.295820,0.012707,0.092279,-0.030165,151,0.503311,1.168351
1,2,"[emotional_response_retail_1, institutional_ac...",1.145822,-1.484505,0.232382,-0.048062,6,0.500000,0.474709


In [22]:
trades = read_parquet_any(rp.data("stage4_trades.parquet"))
print(trades.shape)

# Exit reasons by combination
trades.groupby(["combo_idx", "exit_reason"]).size().rename("n").reset_index().sort_values(["combo_idx", "n"], ascending=[True, False]).head(30)


(17686, 9)


,combo_idx,exit_reason,n
0,1,time_stop,3315
1,1,trigger,737
2,2,time_stop,3
3,2,trigger,3
4,3,time_stop,101
5,3,trigger,50
6,4,time_stop,5058
7,4,trigger,511
8,5,time_stop,7031
9,5,trigger,877


In [23]:
# Trade return distribution (OOS trades are typically tagged the same way; this file includes all trades collected)
trades["return_pct"].describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99])


count    17686.000000
mean         0.002868
std          0.034168
min         -0.292831
1%          -0.101555
5%          -0.044042
10%         -0.028264
50%          0.000000
90%          0.037580
95%          0.056932
99%          0.102336
max          0.375939
Name: return_pct, dtype: float64